# EDA: Evaporator Fouling

## Data dictionary
Same 24 sensor columns + `Datetime` as every Simulated-dataset file — see notebook 01
(undercharge) for the full column-by-column breakdown; not repeated here to avoid
duplication drift across notebooks. Same caveat carries forward: the three `_PRES`
columns are internally consistent but miscalibrated ~14-15x vs. real R410A physics —
relative signals only.

## Why this fault matters as a comparison point

Condenser fouling (notebook 03) established a clean pattern: monotonic, accelerating
signal on the condenser's own sensors, no stage-2 filtering needed, but a weak/muddy
signal on `RTU_TOT_CAPA` at adjacent severities (30 vs 40%, d=-0.037) — plausibly
because capacity also depends on evaporator/airside conditions the condenser doesn't
touch.

Evaporator fouling is the same physical mechanism (heat-transfer surface degradation)
but on the **opposite side** of the refrigerant loop — the coil that absorbs heat from
the return/zone air, rather than the one rejecting it outdoors.

## Hypothesis (before looking at any data)

- Expect the mirror-image pattern of condenser fouling: evaporator-side signals
  (`RTU_REFG_SUCT_PRES`/`_TEMP`, and possibly `RTU_SA_TEMP`/`RTU_RA_TEMP` since the
  evaporator directly conditions supply air) should move cleanly and monotonically,
  the same way condenser-side signals did for that fault.
- Real, falsifiable prediction carried over from notebook 03: since the evaporator is
  physically closer to the capacity-relevant airside process than the condenser is,
  `RTU_TOT_CAPA` might show a *stronger*, cleaner relationship to evaporator fouling
  severity than it did to condenser fouling — this is a genuine guess, not known yet,
  and should be checked against the data rather than assumed to confirm the pattern.
- Watch for whether this fault needs stage-2 filtering at all, same open question as
  every fault so far — no assumption either way until checked.

## Columns being checked: evaporator-side signals

- `RTU_REFG_SUCT_PRES` — suction line pressure (compressor inlet, low-pressure side).
  This line runs directly from the evaporator outlet, so a fouled evaporator that
  can't absorb heat properly should show up here first.
- `RTU_REFG_SUCT_TEMP` — suction line temperature, same physical location/reasoning
  as above.
- `RTU_SA_TEMP` — supply air temperature, i.e. the air actually leaving the unit after
  passing over the evaporator coil. A fouled evaporator should struggle to cool this
  air as effectively, so supply air temp should rise if the fault behaves as expected.
- `RTU_TOT_CAPA` — included again as the cross-fault comparison point carried over
  from condenser fouling (notebook 03), to test the hypothesis that capacity might
  respond more strongly here, given the evaporator's more direct physical link to the
  airside cooling process.

In [1]:
import pandas as pd

files = {
    "baseline": "../data/raw/RTU_sim_baseline.csv",
    "evapfouling10": "../data/raw/RTU_sim_evapfouling10.csv",
    "evapfouling20": "../data/raw/RTU_sim_evapfouling20.csv",
    "evapfouling30": "../data/raw/RTU_sim_evapfouling30.csv",
    "evapfouling40": "../data/raw/RTU_sim_evapfouling40.csv",
    "evapfouling50": "../data/raw/RTU_sim_evapfouling50.csv",
}

dfs = {label: pd.read_csv(fname) for label, fname in files.items()}

for _label, df in dfs.items():
    df["Datetime"] = pd.to_datetime(df["Datetime"])

for label, df in dfs.items():
    print(f"{label}: shape={df.shape}, missing_values={df.isna().sum().sum()}")

baseline: shape=(143941, 25), missing_values=0
evapfouling10: shape=(143941, 25), missing_values=0
evapfouling20: shape=(143941, 25), missing_values=0
evapfouling30: shape=(143941, 25), missing_values=0
evapfouling40: shape=(143941, 25), missing_values=0
evapfouling50: shape=(143941, 25), missing_values=0


In [2]:
import sys
from pathlib import Path

# ml/ is not an installed package (see ml/pyproject.toml: package-mode = false —
# deliberate, since ml/ isn't a deployed service). Notebooks live in ml/notebooks/,
# so add the ml/ root to sys.path to make `from src.features...` imports work
# consistently, the same way pytest's pythonpath = ["."] setting already does
# for the test suite.
ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from src.features.effect_size import cohens_d  # noqa: E402
from src.features.filtering import stage2_only  # noqa: E402

## Load confirmed

All 6 files: shape=(143941, 25), 0 missing values — consistent with every Simulated-
dataset file examined across all fault types so far. `Datetime` converted to
`datetime64[ns]` at load time this notebook, learning from a real hang encountered in
notebook 03 caused by leaving it as `object` dtype through a filtering step.

In [3]:
evap_cols = ["RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_SA_TEMP", "RTU_TOT_CAPA"]

severity_order = ["baseline", "evapfouling10", "evapfouling20", "evapfouling30", "evapfouling40", "evapfouling50"]

summary = pd.DataFrame({
    label: df[evap_cols].mean()
    for label, df in dfs.items()
}).T.loc[severity_order]

summary

,RTU_REFG_SUCT_PRES,RTU_REFG_SUCT_TEMP,RTU_SA_TEMP,RTU_TOT_CAPA
baseline,1.564757e+07,58.343190,56.183938,11801.510557
evapfouling10,1.529189e+07,56.860743,54.467484,11477.454833
evapfouling20,1.489607e+07,55.208372,52.624689,11075.860923
evapfouling30,1.440436e+07,53.114681,50.329509,10647.883778
evapfouling40,1.378777e+07,50.401660,47.434831,10195.372506
evapfouling50,1.309401e+07,47.291231,44.255754,9619.870616


## Finding: evaporator fouling hits capacity far harder than condenser fouling did

All four evaporator-side signals are monotonic across every severity, no exceptions:

| Severity | SUCT_PRES | SUCT_TEMP | SA_TEMP | TOT_CAPA |
|---|---|---|---|---|
| 10% | -2.27% | -2.54% | -3.06% | -2.75% |
| 20% | -4.80% | -5.37% | -6.33% | -6.15% |
| 30% | -7.95% | -8.96% | -10.42% | -9.78% |
| 40% | -11.89% | -13.61% | -15.57% | -13.61% |
| 50% | -16.32% | -18.94% | -21.23% | -18.49% |

**Capacity's drop here (-18.49% at 50%) is roughly 7x larger than condenser fouling's
drop at the same severity (-2.61% at 50%, unfiltered).** This confirms the hypothesis
stated before looking at any data: the evaporator coil does the actual cooling work
`RTU_TOT_CAPA` measures, so fouling here degrades capacity far more directly than
fouling the condenser (which only affects heat rejection *after* cooling has already
happened at the evaporator).

**All four signals also accelerate slightly with severity** — e.g. SA_TEMP's own
step-to-step deltas: -3.06, -3.28, -4.09, -5.15, -5.66 — each 10% increment moves the
signal more than the last, the same qualitative pattern condenser fouling showed on
its own subsystem's signals. Makes physical sense here too: more surface fouling
compounds an already-degraded heat-transfer surface.

**Open question to check next**: is this monotonic pattern still clean after stage-2
filtering, or does the staging confound (seen in undercharge/overcharge, absent in
condenser fouling) show up here too? Not assumed either way.

In [4]:


stage2_dfs = {label: stage2_only(df) for label, df in dfs.items()}

stage2_summary = pd.DataFrame({
    label: df[evap_cols].mean()
    for label, df in stage2_dfs.items()
}).T.loc[severity_order]

stage2_pct_change = (stage2_summary / stage2_summary.loc["baseline"] - 1) * 100
stage2_pct_change

,RTU_REFG_SUCT_PRES,RTU_REFG_SUCT_TEMP,RTU_SA_TEMP,RTU_TOT_CAPA
baseline,0.000000,0.000000,0.000000,0.000000
evapfouling10,-3.500732,-3.922537,-4.166670,-1.211999
evapfouling20,-6.730622,-7.554916,-8.095694,-4.436551
evapfouling30,-10.703450,-12.144875,-13.039938,-7.889561
evapfouling40,-15.400100,-17.789657,-19.103675,-11.958085
evapfouling50,-19.766329,-23.251904,-25.008307,-18.594042


## Stage-2 filtering: clean here, unlike condenser fouling

Unlike condenser fouling (notebook 03), where stage-2 filtering introduced a real
non-monotonic dip in `RTU_TOT_CAPA` (30%→40% reversing direction, later confirmed via
Cohen's d as a genuine near-zero effect between those two severities), **every signal
here stays cleanly monotonic after filtering, with no exceptions**:

| Severity | SUCT_PRES | SUCT_TEMP | SA_TEMP | TOT_CAPA |
|---|---|---|---|---|
| 10% | -3.50% | -3.92% | -4.17% | -1.21% |
| 20% | -6.73% | -7.55% | -8.10% | -4.44% |
| 30% | -10.70% | -12.14% | -13.04% | -7.89% |
| 40% | -15.40% | -17.79% | -19.10% | -11.96% |
| 50% | -19.77% | -23.25% | -25.01% | -18.59% |

Filtering *sharpens* the signal (larger magnitude at every severity vs. the unfiltered
table) without ever breaking the monotonic ordering — the opposite of what happened
with condenser fouling's capacity signal.

**Practical takeaway, now genuinely testable rather than assumed**: evaporator fouling
looks like the "easy" fault of the four examined so far — every candidate feature is
monotonic and well-separated at every severity, both filtered and unfiltered. If this
holds up under a real Cohen's d check between adjacent severities (30 vs 40%, the same
pair that broke down for condenser fouling), evaporator fouling should be comfortably
classifiable even at fine severity resolution — unlike condenser fouling, where capacity
specifically loses discriminating power at mid-range severities.

In [5]:


d_30_vs_40 = cohens_d(
    stage2_dfs["evapfouling30"]["RTU_TOT_CAPA"],
    stage2_dfs["evapfouling40"]["RTU_TOT_CAPA"],
)
print(f"Cohen's d, evapfouling30 vs evapfouling40 (RTU_TOT_CAPA): {d_30_vs_40:.3f}")

Cohen's d, evapfouling30 vs evapfouling40 (RTU_TOT_CAPA): 1.376


## Cohen's d confirms: evaporator fouling's capacity signal is strong at every severity

`evapfouling30` vs `evapfouling40` (stage-2 `RTU_TOT_CAPA`): **d = 1.376** — large,
by the same convention used throughout this project (>0.8 = large). This is the exact
opposite of condenser fouling's result at the equivalent pair (d = -0.037, negligible).

**Conclusion**: evaporator fouling produces a strong, monotonic, well-separated signal
on `RTU_TOT_CAPA` at every severity level checked so far, including the specific
adjacent-severity pair that exposed a real weakness in condenser fouling's capacity
signal. This is consistent with the physical hypothesis stated at the start of this
notebook — the evaporator does the actual cooling work capacity measures, so fouling
here degrades it far more directly and cleanly than fouling the condenser does.

**Two-fault comparison, now with real numbers behind it, not impressions**:

| | Condenser fouling | Evaporator fouling |
|---|---|---|
| Own-subsystem signal (pressure/temp) | Clean, monotonic, accelerating | Clean, monotonic, accelerating |
| `RTU_TOT_CAPA` overall | Weak (-2.6% at 50%) | Strong (-18.6% at 50%) |
| `RTU_TOT_CAPA`, 30% vs 40% (Cohen's d) | -0.037 (negligible) | 1.376 (large) |

## Summary: evaporator fouling EDA

**Confirmed hypothesis**: evaporator fouling behaves like condenser fouling in one
respect (clean, monotonic, accelerating signal on its own subsystem's sensors, no
compressor-stage confound needed) but differs sharply in its effect on `RTU_TOT_CAPA`.

**Strongest signals**: `RTU_REFG_SUCT_PRES`, `RTU_REFG_SUCT_TEMP`, `RTU_SA_TEMP` — all
monotonic and well-separated at every severity, filtered or not.

**Capacity behaves very differently here than for condenser fouling**: -18.59% drop at
50% (filtered) vs. condenser fouling's -2.61% (unfiltered) at the same severity — and,
checked directly via Cohen's d rather than assumed, the specific adjacent-severity pair
that broke down for condenser fouling (30% vs 40%, d=-0.037, negligible) is large and
clean here (d=1.376). Confirms the physical reasoning: the evaporator does the actual
cooling work capacity measures, so fouling it degrades capacity far more directly than
fouling the condenser does.

**Practical implication for modeling**: evaporator fouling looks like the most cleanly
classifiable fault of the four examined so far — every candidate feature, including
capacity, is strongly separated at every severity level.

**Open question carried forward**: two of six Simulated-dataset faults now show
"heat-transfer surface degradation" (condenser, evaporator), with genuinely different
capacity behavior between them. The remaining two — liquid-line restriction and
suction-line restriction — are a third mechanism (a physical restriction/blockage,
not fouling or charge level). No prediction yet for